# 02. Limpieza y Estandarización

Este notebook transforma las fuentes crudas en datasets limpios y alineados al **Modelo Canónico de 33 columnas**.

**Operaciones clave:**
1. **Estandarización de nombres**: Mapeo de español/inglés antiguo a los nombres oficiales del modelo.
2. **Unificación de fechas**: Consolidación de componentes de fecha (interno) y parseo de formatos mixtos (externo).
3. **Imputación de nulos**: 
   - Numéricas: `KNNImputer` (k=5) con escalado previo.
   - Categóricas: Moda.
   - **Excepción**: No se imputa `email`.
4. **Validación de rangos**: Corrección de outliers en `lead_time` y `average_daily_rate`.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Configuración de rutas
RAW_PATH = '../data/raw/'
PROCESSED_PATH = '../data/processed/'
os.makedirs(PROCESSED_PATH, exist_ok=True)

# Carga de datos
df_int = pd.read_csv(os.path.join(RAW_PATH, 'internal_bookings.csv'))
df_ext = pd.read_csv(os.path.join(RAW_PATH, 'external_bookings.csv'))

## 1. Estandarización hacia el Modelo Canónico
Definimos el mapeo para asegurar que ambas fuentes tengan las mismas 33 columnas.

In [2]:
# Mapeo para Fuente Externa (Español -> Modelo Canónico)
ext_rename_map = {
    'tipo_alojamiento': 'hotel',
    'dias_anticipacion': 'lead_time',
    'fecha_llegada': 'arrival_date',
    'pais_origen': 'country',
    'segmento_mercado': 'market_segment',
    'categoria_cliente': 'customer_type',
    'tarifa_diaria_promedio': 'average_daily_rate',
    'tipo_deposito': 'deposit_type',
    'estado_reserva': 'reservation_status',
    'fecha_estado_reserva': 'reservation_status_date',
    'nombre_huesped': 'name',
    'correo_electronico': 'email',
    'telefono_contacto': 'phone_number',
    'total_solicitudes_especiales': 'total_special_requests'
}

# Mapeo para Fuente Interna (Correcciones menores -> Modelo Canónico)
int_rename_map = {
    'adr': 'average_daily_rate',
    'phone-number': 'phone_number',
    'total_of_special_requests': 'total_special_requests'
}

df_ext.rename(columns=ext_rename_map, inplace=True)
df_int.rename(columns=int_rename_map, inplace=True)

# Asegurar que todas las columnas numéricas del modelo canónico existan en el DataFrame antes de imputar
# Nota: 'total_special_requests' ya debería estar renombrado correctamente.

## 2. Unificación de Formatos de Fecha
Llevamos todas las fechas al formato estándar `YYYY-MM-DD`.

In [3]:
# --- FUENTE EXTERNA ---
# pd.to_datetime maneja formatos mixtos automáticamente si es posible
df_ext['arrival_date'] = pd.to_datetime(df_ext['arrival_date'], errors='coerce').dt.strftime('%Y-%m-%d')
df_ext['reservation_status_date'] = pd.to_datetime(df_ext['reservation_status_date'], errors='coerce').dt.strftime('%Y-%m-%d')

# --- FUENTE INTERNA ---
# La interna tiene año, mes (nombre) y día separados.
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}

df_int['month_num'] = df_int['arrival_date_month'].map(month_map)
df_int['arrival_date'] = pd.to_datetime(pd.DataFrame({
    'year': df_int['arrival_date_year'],
    'month': df_int['month_num'],
    'day': df_int['arrival_date_day_of_month']
})).dt.strftime('%Y-%m-%d')

df_int['reservation_status_date'] = pd.to_datetime(df_int['reservation_status_date'], errors='coerce').dt.strftime('%Y-%m-%d')

# Eliminamos columnas de fecha redundantes en la interna para ajustarnos a las 33 del modelo
cols_to_drop = ['arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month', 'arrival_date_week_number', 'month_num']
df_int.drop(columns=cols_to_drop, inplace=True)

print("[OK] Fechas unificadas en formato YYYY-MM-DD")

[OK] Fechas unificadas en formato YYYY-MM-DD


## 3. Imputación de Valores Nulos
Aplicamos la estrategia diferencial solicitada.

In [4]:
def impute_data(df):
    df_work = df.copy()
    
    # A. Variables Categóricas (Moda)
    cat_cols = ['market_segment', 'deposit_type', 'country', 'hotel', 'customer_type', 'meal']
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df_work[cat_cols] = cat_imputer.fit_transform(df_work[cat_cols])
    
    # B. Variables Numéricas (KNN)
    # Columnas numéricas del modelo canónico con nulos potenciales
    num_cols = ['lead_time', 'average_daily_rate', 'total_special_requests', 
                'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children']
    
    # Escalado necesario para KNN
    scaler = MinMaxScaler()
    df_scaled = scaler.fit_transform(df_work[num_cols])
    
    knn_imputer = KNNImputer(n_neighbors=5)
    df_imputed_scaled = knn_imputer.fit_transform(df_scaled)
    
    # Invertir escalado y asignar
    df_work[num_cols] = scaler.inverse_transform(df_imputed_scaled)
    
    return df_work

print("Imputando fuente interna...")
df_int_clean = impute_data(df_int)

print("Imputando fuente externa...")
df_ext_clean = impute_data(df_ext)

print("[OK] Imputación completada (KNN para numéricas, Moda para categóricas). Email intacto.")

Imputando fuente interna...
Imputando fuente externa...
[OK] Imputación completada (KNN para numéricas, Moda para categóricas). Email intacto.


## 4. Validación de Rangos y Homologación
Aseguramos que los valores sean lógicos y que las categorías coincidan.

In [5]:
for df in [df_int_clean, df_ext_clean]:
    # Capar lead_time a 365 días
    df['lead_time'] = df['lead_time'].clip(0, 365)
    # Capar ADR a 1000
    df['average_daily_rate'] = df['average_daily_rate'].clip(0, 1000)
    
    # Homologación de texto (strip y title para consistencia)
    df['reservation_status'] = df['reservation_status'].str.strip().str.title()
    df['deposit_type'] = df['deposit_type'].str.strip().str.title()

print("[OK] Rangos validados y texto homologado.")

[OK] Rangos validados y texto homologado.


## 5. Verificación Final y Exportación
Validamos que ambos datasets tengan las 33 columnas del modelo canónico.

In [6]:
canonical_columns = [
    'hotel', 'is_canceled', 'lead_time', 'arrival_date', 'stays_in_weekend_nights', 
    'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 
    'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 
    'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 
    'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 
    'customer_type', 'average_daily_rate', 'required_car_parking_spaces', 
    'total_special_requests', 'reservation_status', 'reservation_status_date', 
    'name', 'email', 'phone_number', 'credit_card'
]

# Reordenar y asegurar integridad
df_int_clean = df_int_clean[canonical_columns]
df_ext_clean = df_ext_clean[canonical_columns]

print(f"Columnas Internas: {len(df_int_clean.columns)} | Columnas Externas: {len(df_ext_clean.columns)}")

# Guardar
df_int_clean.to_csv(os.path.join(PROCESSED_PATH, 'internal_clean.csv'), index=False)
df_ext_clean.to_csv(os.path.join(PROCESSED_PATH, 'external_clean.csv'), index=False)

print(f"[RESULTADO] Archivos limpios guardados en {PROCESSED_PATH}")

Columnas Internas: 33 | Columnas Externas: 33
[RESULTADO] Archivos limpios guardados en ../data/processed/
